# 🎙️ Voice Assistant — Colab Edition
**Advanced Level — Project 11**

Skills: Speech Recognition • Text-to-Speech

This notebook builds a voice assistant that:
1. Records your voice using your browser's microphone (via JavaScript — Colab has no direct mic access)
2. Transcribes it to text with `SpeechRecognition`
3. Decides on a response (simple rule-based commands, or an LLM for open-ended chat)
4. Speaks the response back to you with `gTTS` (Google Text-to-Speech)

**Note:** microphone recording requires you to grant browser mic permission when prompted, and works in Colab's hosted runtime (not all custom/local runtimes support the JS audio bridge).

## 1. Install dependencies

In [1]:
!pip install -q SpeechRecognition gTTS pydub
!apt-get -qq install -y ffmpeg > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


## 2. Imports

In [2]:
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import io
from pydub import AudioSegment
import datetime
import random

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


## 3. Record audio from your microphone
This uses a small JavaScript snippet to access your browser's mic (Colab notebooks can't access hardware directly). Click the record button that appears, speak, then click it again to stop.

In [3]:
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

var record = time => new Promise(async resolve => {
  let stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  let recorder = new MediaRecorder(stream);
  let chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();

  const btn = document.createElement('button');
  btn.textContent = '⏹ Stop recording';
  btn.style = 'font-size:16px;padding:8px 16px;';
  document.body.appendChild(btn);

  await new Promise(res => { btn.onclick = res; });
  recorder.stop();
  btn.remove();

  await new Promise(res => recorder.onstop = res);
  stream.getTracks().forEach(t => t.stop());

  let blob = new Blob(chunks);
  let reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
"""

def record_audio(filename='input.wav'):
    display(Javascript(RECORD_JS))
    print("🔴 Click the button below, speak, then click it again to stop...")
    data_url = eval_js('record()')
    _, b64 = data_url.split(',', 1)
    audio_bytes = b64decode(b64)

    # Convert whatever the browser gave us (usually webm/ogg) into a clean WAV
    audio = AudioSegment.from_file(io.BytesIO(audio_bytes))
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(filename, format='wav')
    print(f"✅ Saved recording to {filename}")
    return filename

## 4. Speech-to-text (transcription)

In [4]:
recognizer = sr.Recognizer()

def transcribe_audio(filename='input.wav'):
    with sr.AudioFile(filename) as source:
        recognizer.adjust_for_ambient_noise(source, duration=0.3)
        audio_data = recognizer.record(source)
    try:
        text = recognizer.recognize_google(audio_data)
        print(f"🗣️ You said: {text}")
        return text
    except sr.UnknownValueError:
        print("❌ Couldn't understand the audio — try speaking more clearly or reducing background noise.")
        return ""
    except sr.RequestError as e:
        print(f"❌ Speech recognition service error: {e}")
        return ""

## 5. Text-to-speech (spoken reply)

In [5]:
def speak(text, filename='reply.mp3'):
    print(f"🤖 Assistant: {text}")
    tts = gTTS(text=text, lang='en')
    tts.save(filename)
    display(Audio(filename, autoplay=True))

## 6. Decide how to respond
A simple rule-based command router. Good enough for a classic "assistant" demo — ask the time, get a joke, get a greeting, etc. Extend `COMMANDS` with your own intents.

In [6]:
JOKES = [
    "Why do programmers prefer dark mode? Because light attracts bugs.",
    "I told my computer I needed a break, and it said no problem — it froze immediately.",
    "Why did the developer go broke? Because he used up all his cache.",
]

def get_response(text):
    t = text.lower().strip()

    if not t:
        return "I didn't catch that — could you say it again?"
    if 'time' in t:
        return f"It's currently {datetime.datetime.now().strftime('%I:%M %p')}."
    if 'date' in t or 'today' in t:
        return f"Today is {datetime.datetime.now().strftime('%A, %B %d, %Y')}."
    if 'joke' in t:
        return random.choice(JOKES)
    if 'your name' in t:
        return "I'm your Colab voice assistant. Nice to meet you!"
    if any(greet in t for greet in ['hello', 'hi ', 'hey']):
        return "Hello! How can I help you today?"
    if 'thank' in t:
        return "You're welcome!"
    if 'bye' in t or 'exit' in t or 'stop' in t:
        return "Goodbye! Talk soon."

    return f"You said: {text}. I don't have a specific command for that yet, but you can teach me one!"

### Optional: swap the rule-based brain for an LLM
For open-ended conversation instead of fixed commands, replace `get_response()`'s body with a call to an LLM API (Anthropic, OpenAI, etc.) using the transcribed `text` as the prompt. Keep the rule-based checks above it if you still want fast, free responses for common commands like the time or a joke.

## 7. Put it together — one voice turn

In [7]:
def voice_turn():
    record_audio('input.wav')
    text = transcribe_audio('input.wav')
    reply = get_response(text)
    speak(reply)
    return text, reply

# Run this cell, click record, say something like "what time is it" or "tell me a joke"
# voice_turn()

## 8. Full conversation loop
Keeps taking voice turns until you say "bye", "exit", or "stop".

In [8]:
def run_assistant(max_turns=10):
    speak("Hi, I'm listening. Say something whenever you're ready.")
    for _ in range(max_turns):
        text, reply = voice_turn()
        if any(word in text.lower() for word in ['bye', 'exit', 'stop']):
            break

# Uncomment to start a multi-turn conversation:
# run_assistant()

## 9. Text-only test (no microphone needed)
Useful for quickly testing the response + TTS logic without recording audio each time.

In [9]:
def text_turn(text):
    print(f"🗣️ You typed: {text}")
    reply = get_response(text)
    speak(reply)

text_turn("tell me a joke")

🗣️ You typed: tell me a joke
🤖 Assistant: Why did the developer go broke? Because he used up all his cache.


## Notes
- `recognize_google()` from `SpeechRecognition` uses Google's free Web Speech API — no API key needed, but it's rate-limited and not intended for heavy production use. For production-grade accuracy or higher volume, swap in Google Cloud Speech-to-Text, Whisper, or another paid STT engine.
- `gTTS` requires an internet connection (it calls Google Translate's TTS endpoint) and produces one fixed voice. For offline TTS or custom voices, look at `pyttsx3` (offline, robotic-sounding) or a neural TTS service.
- The JS mic-recording bridge only works in Colab's browser-connected runtime — it won't work if you're connected to a local runtime without a browser mic path.